```perl
 ┌─────────────────────────────────────────────────────────┐
 │           DATA EXTRACTION & INSPECTION                  │
 │  Load Kaggle CSV (25,000 rows x 15 cols)                │
 │  Inspect shape, head, info, nulls                       │
 └──────────────────────┬──────────────────────────────────┘
                        │
                        ▼
 ┌─────────────────────────────────────────────────────────┐
 │           DATA CLEANING                                 │
 │  Standardise names, parse dates, compute slack_time     │
 │  Median/mode imputation, prune post-delivery leakage    │
 └──────────────────────┬──────────────────────────────────┘
                        │
                        ▼
 ┌─────────────────────────────────────────────────────────┐
 │           EXPLORATORY DATA ANALYSIS (EDA)               │
 │  Part 1: Pre-outlier stats & plots                      │
 │  Part 2: Smart outlier removal (selected numeric cols)  │
 │  Part 3: Post-outlier plots, correlations               │
 └──────────────────────┬──────────────────────────────────┘
                        │
                        ▼
 ┌─────────────────────────────────────────────────────────┐
 │           FEATURE ENGINEERING (Pre-Split)               │
 │  Interaction: weight_x_distance, km_per_expected_hr,    │
 │               cost_per_kg                               │
 │  Ordinal/Risk: weather_severity, mode_urgency,          │
 │                schedule_risk, vehicle_capacity,         │
 │                vehicle_load_strain                      │
 │  Group Agg: carrier_avg_schedule, carrier_avg_weight,   │
 │             region_avg_distance                         │
 │  Targets: delay_hours, delayed (binary)                 │
 └──────────────────────┬──────────────────────────────────┘
                        │
                        ▼
 ┌─────────────────────────────────────────────────────────┐
 │           FEATURE SELECTION                             │
 │  Drop correlated: delivery_cost, cost_per_km,           │
 │                  expected_time_hrs_num, delivery_rating │
 └──────────────────────┬──────────────────────────────────┘
                        │
                        ▼
 ┌─────────────────────────────────────────────────────────┐
 │           TRAIN / TEST SPLIT                            │
 │  Stratified split on delayed target                     │
 │  Align delivery_id with train/test for traceability     │
 └──────────────────────┬──────────────────────────────────┘
                        │
               ┌────────┴────────┐
               ▼                 ▼
 ┌──────────────────┐  ┌──────────────────┐
 │ Post-Split 3a    │  │ Post-Split 3b    │
 │ One-Hot Encoding │  │ One-Hot + Scaling│
 │ (tree models)    │  │ (linear/SVM)     │
 └────────┬─────────┘  └────────┬─────────┘
          └────────┬────────────┘
                   ▼
 ┌─────────────────────────────────────────────────────────┐
 │           BASELINE MODELS                               │
 │  Logistic Regression (classification baseline)          │
 │  Linear Regression (regression baseline)                │
 └──────────────────────┬──────────────────────────────────┘
                        │
                        ▼
 ┌─────────────────────────────────────────────────────────┐
 │     STAGE 1: BINARY CLASSIFICATION — delayed (Yes/No)   │
 │                                                         │
 │  Train & evaluate 7 classifiers:                        │
 │    Logistic Reg, Decision Tree, Random Forest,          │
 │    AdaBoost, XGBoost, LightGBM, SVM, Naive Bayes        │
 │                                                         │
 │  Compare all → Best: Random Forest                      │
 │  Save best_classification_random_forest.pkl             │
 └──────────────────────┬──────────────────────────────────┘
                        │
                        ▼
 ┌─────────────────────────────────────────────────────────┐
 │     STAGE 2: TWO-STAGE PIPELINE (Classify → Severity)   │
 │                                                         │
 │  Filter to predicted-delayed rows only                  │
 │                                                         │
 │  ┌─────────────────────┐  ┌──────────────────────────┐  │
 │  │ Option 1 (chosen)   │  │ Option 2 (comparison)    │  │
 │  │ Severity Classif.   │  │ Ordinal Regression       │  │
 │  │ RF → Short/Med/Long │  │ Frank & Hall cumulative  │  │
 │  │ (1-2h / 3-5h / 6+h) │  │ threshold RFs → hours    │  │
 │  └─────────┬───────────┘  └──────────┬───────────────┘  │
 │            │   Best: Option 1        │                  │
 └────────────┼─────────────────────────┼──────────────────┘
              │                         │
              ▼                         │
 ┌─────────────────────────────────────────────────────────┐
 │           END-TO-END EVALUATION                         │
 │  Chain Stage 1 (binary) + Stage 2 (severity) on full    │
 │  test set; non-delayed → "No Delay" class               │
 │  Metrics: accuracy, F1, confusion matrices              │
 └──────────────────────┬──────────────────────────────────┘
                        │
                        ▼
 ┌─────────────────────────────────────────────────────────┐
 │           MODEL PERSISTENCE                             │
 │  Save best_severity_random_forest.pkl + metadata JSON   │
 │  Reload & verify both stage-1 and stage-2 models        │
 └──────────────────────┬──────────────────────────────────┘
                        │
                        ▼
 ┌─────────────────────────────────────────────────────────┐
 │           DATABASE & PREDICTIONS                        │
 │  Run two-stage predictions on train + test sets         │
 │  Save to delivery_predictions.db (SQLite)               │
 │  Build hist_* and daily_* summary tables                │
 │  Write prediction CSVs                                  │
 │  Verify: assert row counts, IDs, severity labels        │
 └─────────────────────────────────────────────────────────┘
 ```

**Pipeline summary:**

- Extract raw logistics data (25K orders)
- Clean and impute, removing post-delivery leakage columns
- EDA with outlier detection and removal
- Engineer 10+ derived features (interaction, ordinal, group-aggregate)
- Select features by dropping correlated redundancies
- Split stratified train/test, then two encoding paths (tree vs linear)
- Stage 1 — train 8 binary classifiers on delayed, pick Random Forest
- Stage 2 — on predicted-delayed subset, train severity classifier (Short/Medium/Long); compare against ordinal regression alternative
- Persist both models (.pkl + metadata JSON), reload and verify
- Generate full predictions, save to SQLite DB with summary tables, write CSVs, and assert correctness